# 🐟 Fish-Speech LoRA Fine-Tuning: Bahasa Indonesia (T4 16GB GPU)

Notebook ini melatih model TTS multilingual **Fish-Speech** menggunakan dataset publik **`X-lord/Dataset-Text-To-Speech-Indonesia`** (4.531 sampel, ~16.4 jam audio narasi Bahasa Indonesia).

### ⚡ Persyaratan:
- Runtime: **GPU (T4 16GB VRAM - Gratis)** di Google Colab.
- Pastikan menu: **Runtime > Change runtime type > T4 GPU** sudah terpilih.

In [ ]:
# 1. Verifikasi Akses GPU Colab
!nvidia-smi

In [ ]:
# 2. Clone Fish-Speech Repository & Install Dependencies
!git clone https://github.com/fishaudio/fish-speech.git
%cd fish-speech
!pip install -e .
!pip install datasets huggingface_hub soundfile scipy pyarrow pydub lightning hydra-core

In [ ]:
# 3. Download Base Model Checkpoint (openaudio-s1-mini)
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="fishaudio/openaudio-s1-mini",
    local_dir="checkpoints/openaudio-s1-mini",
    ignore_patterns=["*.bin", "*.msgpack"]
)
print("✅ Base model openaudio-s1-mini berhasil diunduh!")

In [ ]:
# 4. Download & Preprocess Dataset Indonesia (X-lord/Dataset-Text-To-Speech-Indonesia)
# Mengunduh langsung di Colab (kecepatan network Google > 100 MB/s!)
import os
import io
import soundfile as sf
import numpy as np
from scipy import signal
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
import pyarrow.parquet as pq

output_dir = Path("data/Speaker_Indonesia")
output_dir.mkdir(parents=True, exist_ok=True)

api = HfApi()
dataset_repo = "X-lord/Dataset-Text-To-Speech-Indonesia"
files = [f for f in api.list_repo_files(dataset_repo, repo_type="dataset") if f.startswith("data/train-") and f.endswith(".parquet")]

TARGET_SR = 24000
MAX_SAMPLES = 500  # Ubah ke None untuk seluruh 4.531 file
count = 0

print(f"Mengunduh {len(files)} file partisi dari HuggingFace...")
for pq_file in files:
    if MAX_SAMPLES and count >= MAX_SAMPLES:
        break
    p_local = hf_hub_download(dataset_repo, pq_file, repo_type="dataset")
    table = pq.read_table(p_local)
    for i in range(table.num_rows):
        if MAX_SAMPLES and count >= MAX_SAMPLES:
            break
        row = {c: table[c][i].as_py() for c in table.column_names}
        text = str(row.get("text", "")).strip()
        audio_obj = row.get("audio")
        if not text or not audio_obj:
            continue
        b = audio_obj["bytes"] if isinstance(audio_obj, dict) else audio_obj
        data, sr = sf.read(io.BytesIO(b), dtype="float32")
        if data.ndim > 1:
            data = np.mean(data, axis=1)
        if sr != TARGET_SR:
            data = signal.resample(data, int(len(data) * TARGET_SR / sr))
        # Normalize
        mx = np.max(np.abs(data))
        if mx > 1e-5:
            data = data / mx * 0.90
        count += 1
        wav_name = f"segment_{count:05d}.wav"
        lab_name = f"segment_{count:05d}.lab"
        sf.write(str(output_dir / wav_name), data.astype(np.float32), TARGET_SR, format="WAV", subtype="PCM_16")
        with open(output_dir / lab_name, "w", encoding="utf-8") as f:
            f.write(text)
        if count % 50 == 0:
            print(f"Progress: {count} file audio dan transkrip siap!")

print(f"🎉 Total {count} sampel audio 24kHz dan .lab berhasil disiapkan!")

In [ ]:
# 5. Ekstraksi Semantic VQ Tokens
!python tools/vqgan/extract_vq.py "data/Speaker_Indonesia" \
    --num-workers 2 \
    --batch-size 16 \
    --config-name "modded_dac_vq" \
    --checkpoint-path "checkpoints/openaudio-s1-mini/codec.pth"
print("✅ Semantic VQ extraction selesai!")

In [ ]:
# 6. Build Protobuf Dataset untuk Training
!python tools/llama/build_dataset.py \
    --input "data/Speaker_Indonesia" \
    --output "data/protos" \
    --text-extension .lab \
    --num-workers 4
print("✅ Protobuf dataset siap untuk training!")

In [ ]:
# 7. Jalankan LoRA Fine-Tuning pada T4 GPU
!python fish_speech/train.py \
    --config-name "text2semantic_finetune_lora" \
    project="indonesia-tts" \
    model.lora_config.r=8 \
    model.lora_config.lora_alpha=16 \
    trainer.max_steps=200 \
    trainer.val_check_interval=50
print("🎉 Training LoRA selesai!")

In [ ]:
# 8. Test Sintesis Suara Bahasa Indonesia dengan Model Baru
!python tools/llama/generate.py \
    --text "Halo semua! Selamat datang di episode terbaru podcast kita. Hari ini kita membahas kecerdasan buatan." \
    --checkpoint-path "checkpoints/openaudio-s1-mini" \
    --output "test_indonesia.wav"

from IPython.display import Audio
Audio("test_indonesia.wav")

In [ ]:
# 9. Download Checkpoint ke PC Lokal Anda
!zip -r indonesia-tts-lora.zip results/indonesia-tts/checkpoints/
from google.colab import files
files.download("indonesia-tts-lora.zip")
print("File checkpoint siap dipindahkan ke folder services/fish-speech/checkpoints di PC lokal Anda!")